# HAZEM ELSHAFEY | Pandas for AI

**HBE_2206 — Python for AI · Library Series**

> Visual identity: charcoal `#191919` · yellow `#FFDC52` · cream `#FAF8F0` · orange `#FF6B35`.

Pandas helps load, inspect, clean, transform and summarize tabular data. This notebook builds on Python and NumPy and prepares you for exploratory data analysis and machine learning.

**Learning outcomes**
- Understand `Series` and `DataFrame`.
- Load and inspect CSV data.
- Select rows and columns with `loc` and `iloc`.
- Clean missing values and duplicates.
- Transform columns, filter rows, sort, group, merge and export.
- Build a compact data-cleaning workflow.

If needed, install dependencies in your environment with `pip install pandas numpy` (or use a managed environment such as Colab).

In [ ]:
import numpy as np
import pandas as pd
print('Pandas version:', pd.__version__)
print('NumPy version:', np.__version__)

## 1. Series and DataFrame

A `Series` is a one-dimensional labeled array. A `DataFrame` is a two-dimensional table with labeled rows and columns. DataFrames can contain different data types in different columns.

In [ ]:
scores = pd.Series([85, 92, 78], index=['Mona', 'Ali', 'Omar'], name='Score')
print(scores)
print('Ali score:', scores['Ali'])

df = pd.DataFrame({'Name': ['Mona', 'Ali', 'Omar'], 'Age': [21, 22, 20], 'Score': [85, 92, 78]})
print(df)

**Expected output**
```text
Mona    85
Ali     92
Omar    78
Name: Score, dtype: int64
Ali score: 92
   Name  Age  Score
0  Mona   21     85
1   Ali   22     92
2  Omar   20     78
```

## 2. Create a DataFrame and inspect it

Useful inspection tools: `head`, `tail`, `shape`, `columns`, `dtypes`, `info`, `describe`, and `isna().sum()`. `info()` prints a summary and typically returns `None`.

In [ ]:
df = pd.DataFrame({
    'Name': ['Mona', 'Ali', 'Omar', 'Sara'],
    'Age': [21, 22, np.nan, 24],
    'Salary': [9000, 12000, 10000, np.nan],
    'Department': ['AI', 'Data', 'AI', 'Data']
})
print(df.head(2))
print('shape:', df.shape)
print('columns:', df.columns.tolist())
print('dtypes:\n', df.dtypes)
print('missing by column:\n', df.isna().sum())

**Expected output**
```text
   Name   Age   Salary Department
0  Mona  21.0   9000.0         AI
1   Ali  22.0  12000.0       Data
shape: (4, 4)
columns: ['Name', 'Age', 'Salary', 'Department']
dtypes:
 Name           object
Age           float64
Salary        float64
Department     object
missing by column:
 Name          0
Age           1
Salary        1
Department    0
```

## 3. Read and write CSV files

`pd.read_csv()` loads a CSV into a DataFrame. `to_csv()` writes a DataFrame; `index=False` avoids adding the DataFrame index as an extra column. File paths depend on your notebook's working directory.

In [ ]:
# Example: load a local CSV file
# data = pd.read_csv('data.csv')
# print(data.head())

# Small self-contained export/import example
sample = pd.DataFrame({'feature': [1, 2, 3], 'target': [0, 1, 1]})
sample.to_csv('sample_data.csv', index=False)
loaded = pd.read_csv('sample_data.csv')
print(loaded)

**Expected output**
```text
   feature  target
0        1       0
1        2       1
2        3       1
```

For larger or important files, verify the path, encoding, delimiter and data types after loading.

## 4. Select columns, rows, `loc` and `iloc`

- `df['Age']` selects one column as a Series.
- `df[['Name', 'Age']]` selects multiple columns as a DataFrame.
- `loc` uses row/column labels.
- `iloc` uses integer positions.

The index label is not always the same as the row's integer position.

In [ ]:
df = pd.DataFrame({'Name': ['Mona', 'Ali', 'Omar'], 'Age': [21, 22, 20], 'Score': [85, 92, 78]}, index=['s1', 's2', 's3'])
print(df['Name'])
print('\nloc s2:\n', df.loc['s2'])
print('\niloc first row:\n', df.iloc[0])
print('\nselected columns:\n', df[['Name', 'Score']])

**Expected output**
```text
s1    Mona
s2     Ali
s3    Omar
Name: Name, dtype: object

loc s2:
 Name     Ali
Age       22
Score     92
Name: s2, dtype: object

iloc first row:
 Name     Mona
Age        21
Score      85
Name: s1, dtype: object

selected columns:
     Name  Score
s1  Mona     85
s2   Ali     92
s3  Omar     78
```

## 5. Filter rows

Boolean masks select rows meeting conditions. Use `&` for AND and `|` for OR, with each condition in parentheses.

In [ ]:
df = pd.DataFrame({'Name': ['Mona', 'Ali', 'Omar', 'Sara'], 'Age': [21, 22, 20, 24], 'Score': [85, 92, 78, 95]})
print(df[df['Score'] >= 90])
print('\nAge 21–23:\n', df[(df['Age'] >= 21) & (df['Age'] <= 23)])

**Expected output**
```text
   Name  Age  Score
1   Ali   22     92
3   Sara   24     95

Age 21–23:
    Name  Age  Score
0  Mona   21     85
1   Ali   22     92
```

## 6. Missing data: detect, drop and fill

`isna()` marks missing values; `sum()` counts `True` as 1. `dropna()` removes rows by default; `dropna(axis=1)` removes columns with missing values. `subset` limits which columns determine row removal. `fillna()` fills missing values.

**Important:** replacing missing ages with zero is not always statistically appropriate. Choose a strategy based on what the missing value means and the downstream task.

In [ ]:
df = pd.DataFrame({'Name': ['Mona', 'Ali', 'Omar', 'Sara'], 'Age': [21, 22, np.nan, 24], 'Salary': [9000, np.nan, 10000, np.nan]})
print('missing counts:\n', df.isna().sum())
print('\nDrop rows with any missing value:\n', df.dropna())
print('\nDrop rows missing Salary:\n', df.dropna(subset=['Salary']))
print('\nFill Age with mean:\n', df.assign(Age=df['Age'].fillna(df['Age'].mean())))

**Expected output**
```text
missing counts:
 Name      0
Age       1
Salary    2

Drop rows with any missing value:
    Name   Age  Salary
0  Mona  21.0  9000.0

Drop rows missing Salary:
    Name   Age  Salary
0  Mona  21.0  9000.0
2  Omar   NaN 10000.0

Fill Age with mean:
    Name   Age  Salary
0  Mona  21.000000 9000.0
1   Ali  22.000000    NaN
2  Omar  22.333333 10000.0
3  Sara  24.000000    NaN
```

Forward fill (`ffill`) propagates the previous valid value; backward fill (`bfill`) uses the next valid value. These can be useful for ordered time-series data, but can be inappropriate when row order has no meaning.

In [ ]:
series = pd.Series([10, np.nan, np.nan, 40, np.nan])
print('ffill:', series.ffill().tolist())
print('bfill:', series.bfill().tolist())

**Expected output**
```text
ffill: [10.0, 10.0, 10.0, 40.0, 40.0]
bfill: [40.0, 40.0, 40.0, 40.0, nan]
```

## 7. Duplicates and data types

Use `duplicated()` to identify repeated rows and `drop_duplicates()` to remove them. Use `astype()` for deliberate type conversions. Parse dates with `pd.to_datetime()` and inspect conversion errors.

In [ ]:
df = pd.DataFrame({'Name': ['Ali', 'Mona', 'Ali'], 'Score': [90, 85, 90]})
print('duplicate rows:', df.duplicated().tolist())
print('deduplicated:\n', df.drop_duplicates())

raw = pd.Series(['10', '20', 'unknown'])
numeric = pd.to_numeric(raw, errors='coerce')
print('numeric with invalid -> NaN:', numeric.tolist())

**Expected output**
```text
duplicate rows: [False, False, True]
deduplicated:
    Name  Score
0   Ali     90
1  Mona     85
numeric with invalid -> NaN: [10.0, 20.0, nan]
```

## 8. Create and transform columns

You can create columns from calculations, use `.assign()`, and map categories. Prefer clear vectorized operations over row-by-row loops for common transformations.

In [ ]:
df = pd.DataFrame({'Name': ['Mona', 'Ali', 'Omar'], 'Score': [85, 92, 58]})
df['Passed'] = df['Score'] >= 60
df['Score_Percent'] = df['Score'] / 100
print(df)
print(df['Name'].str.upper().tolist())

**Expected output**
```text
   Name  Score  Passed  Score_Percent
0  Mona     85    True           0.85
1   Ali     92    True           0.92
2  Omar     58   False           0.58
['MONA', 'ALI', 'OMAR']
```

## 9. Sorting and value counts

`sort_values()` sorts by one or more columns. `value_counts()` counts occurrences of categorical values and normally sorts by frequency.

In [ ]:
df = pd.DataFrame({'Department': ['AI', 'Data', 'AI', 'Data', 'AI'], 'Score': [85, 92, 78, 88, 95]})
print(df.sort_values('Score', ascending=False))
print('\nDepartment counts:\n', df['Department'].value_counts())

**Expected output**
```text
  Department  Score
4         AI     95
1       Data     92
3       Data     88
0         AI     85
2         AI     78

Department counts:
Department
AI      3
Data    2
Name: count, dtype: int64
```

The exact display of `value_counts()` metadata can vary slightly by Pandas version; the counts are 3 and 2.

## 10. GroupBy and aggregation

`groupby()` follows split–apply–combine: split rows into groups, apply aggregations, and combine the results. Common operations include `mean`, `sum`, `count`, `min`, and `max`.

In [ ]:
df = pd.DataFrame({'Department': ['AI', 'Data', 'AI', 'Data', 'AI'], 'Salary': [9000, 12000, 11000, 10000, 13000]})
print(df.groupby('Department')['Salary'].mean())
print('\nSummary:\n', df.groupby('Department').agg(avg_salary=('Salary', 'mean'), employees=('Salary', 'count')))

**Expected output**
```text
Department
AI      11000.0
Data    11000.0
Name: Salary, dtype: float64

Summary:
              avg_salary  employees
Department
AI              11000.0          3
Data            11000.0          2
```

## 11. Merge and concatenate

`merge()` joins tables using matching keys, similar to a database join. `concat()` stacks DataFrames or Series along an axis. Check key uniqueness and join type when combining real datasets.

In [ ]:
employees = pd.DataFrame({'employee_id': [1, 2, 3], 'Name': ['Mona', 'Ali', 'Sara']})
salaries = pd.DataFrame({'employee_id': [1, 2, 4], 'Salary': [9000, 12000, 15000]})
print(employees.merge(salaries, on='employee_id', how='left'))
print('\nConcatenate rows:\n', pd.concat([employees, employees], ignore_index=True))

**Expected output**
```text
   employee_id  Name   Salary
0            1  Mona   9000.0
1            2   Ali  12000.0
2            3  Sara      NaN

Concatenate rows:
   employee_id  Name
0            1  Mona
1            2   Ali
2            3  Sara
3            1  Mona
4            2   Ali
5            3  Sara
```

## 12. Dates and time-series basics

Use `pd.to_datetime()` to parse dates. Datetime accessors such as `.dt.year` and `.dt.month` expose date parts. Sort by time before applying forward-fill or rolling operations.

In [ ]:
df = pd.DataFrame({'date': ['2026-01-03', '2026-01-01', '2026-01-02'], 'sales': [30, 10, 20]})
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date')
df['month'] = df['date'].dt.month
print(df.to_string(index=False))

**Expected output**
```text
      date  sales  month
2026-01-01     10      1
2026-01-02     20      1
2026-01-03     30      1
```

## 13. Quick exploratory data analysis

Use `describe()` for numeric summaries, `corr()` for numeric correlation, and targeted checks for missing values and duplicates. Correlation does not establish causation, and correlations can be misleading when data quality or sample size is poor.

In [ ]:
df = pd.DataFrame({'hours_studied': [1, 2, 3, 4, 5], 'score': [50, 55, 65, 70, 80]})
print(df.describe().round(2))
print('\nCorrelation:\n', df.corr().round(2))

**Expected output**
```text
       hours_studied  score
count           5.00   5.00
mean            3.00  64.00
std             1.58  12.04
min             1.00  50.00
25%             2.00  55.00
50%             3.00  65.00
75%             4.00  70.00
max             5.00  80.00

Correlation:
                hours_studied  score
hours_studied           1.00   0.99
score                   0.99   1.00
```

## 14. Mini-project: clean a small employee dataset

This workflow checks missingness, fills numeric data with the median, removes duplicates, derives a monthly salary column, and summarizes by department. In real ML work, learn imputation parameters from training data only and apply them to validation/test data.

In [ ]:
df = pd.DataFrame({
    'Name': ['Mona', 'Ali', 'Omar', 'Sara', 'Sara'],
    'Department': ['AI', 'Data', 'AI', 'Data', 'Data'],
    'Age': [21, 22, np.nan, 24, 24],
    'AnnualSalary': [108000, 144000, 120000, np.nan, np.nan]
})
print('Missing before cleaning:\n', df.isna().sum())
df = df.drop_duplicates().copy()
df['Age'] = df['Age'].fillna(df['Age'].median())
df['AnnualSalary'] = df['AnnualSalary'].fillna(df['AnnualSalary'].median())
df['MonthlySalary'] = df['AnnualSalary'] / 12
print('\nClean data:\n', df.to_string(index=False))
print('\nDepartment summary:\n', df.groupby('Department').agg(avg_monthly_salary=('MonthlySalary', 'mean'), employees=('Name', 'count')).round(2))

**Expected output (key results)**
```text
Missing before cleaning:
Name            0
Department      0
Age             1
AnnualSalary    2

Clean data contains 4 unique rows. Missing Age is filled with 22.5; missing AnnualSalary values are filled with the median 120000. MonthlySalary is AnnualSalary / 12.

Department summary:
             avg_monthly_salary  employees
Department
AI                     9666.67          2
Data                  11000.00          2
```

Tip: run the cell to inspect the full formatted table in your notebook.

## 15. Common mistakes and best practices

- `df['column']` returns a Series; `df[['column']]` returns a DataFrame.
- Use `loc` for labels and `iloc` for integer positions.
- Check `df.shape`, `df.dtypes`, and `df.isna().sum()` before analysis.
- Avoid chained assignment; use `.loc[...] = ...` or `.assign(...)` and make explicit copies when needed.
- Do not fill every missing value with zero without understanding its meaning.
- When grouping, merging or dropping duplicates, verify the resulting row count.
- Keep raw data unchanged and save cleaned data to a separate file when appropriate.
- Prevent data leakage: fit preprocessing choices on training data only.

## Practice tasks
1. Create a DataFrame with `Name`, `Age`, and `Score` for five students.
2. Filter students with scores at least 80 and sort descending.
3. Add a `Passed` column using a threshold of 60.
4. Create two missing values and compare `dropna()` with median imputation.
5. Group the data by a category and calculate mean score and count.
6. Export the cleaned table to CSV without the index.

## Reference solution (try the tasks first)

In [ ]:
students = pd.DataFrame({
    'Name': ['Mona', 'Ali', 'Omar', 'Sara', 'Nour'],
    'Age': [21, 22, np.nan, 24, 20],
    'Score': [85, 92, 58, 95, 78],
    'Track': ['AI', 'Data', 'AI', 'Data', 'AI']
})
students['Age'] = students['Age'].fillna(students['Age'].median())
students['Passed'] = students['Score'] >= 60
print(students[students['Score'] >= 80].sort_values('Score', ascending=False))
print('\nGroup summary:\n', students.groupby('Track').agg(mean_score=('Score', 'mean'), students=('Name', 'count')))
students.to_csv('students_clean.csv', index=False)
print('\nSaved students_clean.csv')

## Further learning

Next: combine NumPy arrays with Pandas tables, then learn visualization with Matplotlib/Seaborn and build clean, reproducible data-analysis notebooks.

**HAZEM ELSHAFEY · HBE_2206**  
GitHub: https://github.com/HBE-2206/Python-for-AI